# XGBOOST SIMPLE FROM-SCRATCH VERSION + SCIKIT-LEARN STYLE USAGE

## What's in this notebook

- 3\. Simple XGBoost-style regression
- 7\. Real XGBoost
- 11\. Experiment with number of trees
- 12\. Experiment with learning rate
- 13\. Experiment with max depth
- 14\. Important XGBoost parameters
- 15\. XGBoost core math
- 16\. Split gain
- 17\. Core working
- 18\. Gradient boosting vs XGBoost
- 19\. Interview summary

In [ ]:
import numpy as np

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# 1. CREATE DATASET

In [ ]:
X, y = make_regression(
    n_samples=300,
    n_features=2,
    noise=10,
    random_state=42
)

# 2. TRAIN / TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3. SIMPLE XGBOOST-STYLE REGRESSION

A tiny version of XGBoost's core idea, with NumPy only. Every round fits a **stump** (a tree with one split) to the gradients:

- the split is the one with the best **XGBoost gain**:  gain = ½ · [ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ] − γ
- each leaf gets the weight **w = −G / (H + λ)**
- G = sum of gradients, H = sum of Hessians in that leaf. For squared error, gradient = prediction − y and Hessian = 1

Tested on this data: R² ≈ 0.95, the same as scikit-learn's `GradientBoostingRegressor` with stumps.
(An earlier version added one constant per round and never split on X, so it always predicted the mean: R² ≈ 0.)

In [ ]:
class SimpleXGBoost:
    """XGBoost's core idea in miniature (squared error, numpy only).

    Each round fits a stump (a tree with one split) to the gradients:
    - the split is the one with the best XGBoost gain
    - each leaf gets the weight w = -G / (H + lambda)
    For squared error: gradient = prediction - y, Hessian = 1.
    """

    def __init__(self, n_estimators=100, learning_rate=0.3, reg_lambda=1.0, gamma=0.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.stumps = []                       # (feature, threshold, left weight, right weight)

    def _score(self, G, H):
        return G ** 2 / (H + self.reg_lambda)

    def _best_stump(self, X, g, h):
        G, H = g.sum(), h.sum()
        best = None
        for f in range(X.shape[1]):
            order = np.argsort(X[:, f])
            xs, gs, hs = X[order, f], g[order], h[order]
            G_left, H_left = np.cumsum(gs)[:-1], np.cumsum(hs)[:-1]     # rows 0..i go left
            G_right, H_right = G - G_left, H - H_left
            gain = 0.5 * (self._score(G_left, H_left) + self._score(G_right, H_right) - self._score(G, H)) - self.gamma
            gain[xs[:-1] == xs[1:]] = -np.inf     # cannot split between equal values
            i = int(np.argmax(gain))
            if best is None or gain[i] > best[0]:
                threshold = (xs[i] + xs[i + 1]) / 2
                w_left = -G_left[i] / (H_left[i] + self.reg_lambda)
                w_right = -G_right[i] / (H_right[i] + self.reg_lambda)
                best = (gain[i], f, threshold, w_left, w_right)
        return best

    def fit(self, X, y):
        self.base_prediction = y.mean()
        pred = np.full(len(y), self.base_prediction)
        for _ in range(self.n_estimators):
            g = pred - y                       # gradient of squared error
            h = np.ones(len(y))                # Hessian of squared error
            gain, f, t, w_left, w_right = self._best_stump(X, g, h)
            if gain <= 0:                      # no split is worth its gamma penalty: stop
                break
            self.stumps.append((f, t, w_left, w_right))
            pred += self.learning_rate * np.where(X[:, f] <= t, w_left, w_right)
        return self

    def predict(self, X):
        pred = np.full(len(X), self.base_prediction)
        for f, t, w_left, w_right in self.stumps:
            pred += self.learning_rate * np.where(X[:, f] <= t, w_left, w_right)
        return pred

# 4. TRAIN SIMPLE MODEL

```text
Gradient Boosting
       ↓
Sequential trees
       ↓
Correct errors
```

```text
XGBoost
       ↓
Sequential trees
       ↓
Gradient + Hessian
       ↓
Find best split using gain
       ↓
Regularize leaf weights
       ↓
Add tree × learning rate
```

In [ ]:
scratch_model = SimpleXGBoost(n_estimators=100, learning_rate=0.3, reg_lambda=1.0)
scratch_model.fit(X_train, y_train)

# 5. PREDICT

In [ ]:
scratch_pred = scratch_model.predict(
    X_test
)

# 6. EVALUATE

In [ ]:
print(
    "Scratch MSE:",
    mean_squared_error(
        y_test,
        scratch_pred
    )
)

print(
    "Scratch R2:",
    r2_score(
        y_test,
        scratch_pred
    )
)

# 7. REAL XGBOOST

In [ ]:
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    objective="reg:squarederror",
    random_state=42
)

# 8. TRAIN

In [ ]:
model.fit(
    X_train,
    y_train
)

# 9. PREDICT

In [ ]:
pred = model.predict(
    X_test
)

# 10. EVALUATE

In [ ]:
mse = mean_squared_error(
    y_test,
    pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    pred
)

print("XGBoost MSE:", mse)
print("XGBoost RMSE:", rmse)
print("XGBoost R2:", r2)

# 11. EXPERIMENT WITH NUMBER OF TREES

In [ ]:
for n in [10, 50, 100, 200]:

    model = XGBRegressor(
        n_estimators=n,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "n_estimators:",
        n,
        "| R2:",
        score
    )

# 12. EXPERIMENT WITH LEARNING RATE

In [ ]:
for lr in [0.01, 0.05, 0.1, 0.2]:

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "learning_rate:",
        lr,
        "| R2:",
        score
    )

# 13. EXPERIMENT WITH MAX DEPTH

In [ ]:
for depth in [1, 2, 3, 5, 8]:

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    score = model.score(
        X_test,
        y_test
    )

    print(
        "max_depth:",
        depth,
        "| R2:",
        score
    )

# 14. IMPORTANT XGBOOST PARAMETERS

n_estimators

Number of boosting trees.

```text
More trees → more learning steps.
```

learning_rate

Controls the contribution of each tree.

```text
Small learning_rate → usually need more trees.
```

max_depth

Maximum depth of each tree.

```text
Larger → more complex model
Smaller → simpler model
```

min_child_weight

Minimum amount of Hessian weight required  
in a child node.

```text
Higher value → more conservative splitting.
```

subsample

Fraction of training samples used for each tree.

Example:

subsample=0.8

```text
→ use 80% of training data per tree.
```

colsample_bytree

Fraction of features used by each tree.

gamma

Minimum loss reduction required  
to make a split.

```text
Higher gamma → fewer splits.
```

reg_lambda

L2 regularization on leaf weights.

reg_alpha

L1 regularization on leaf weights.

objective

Defines the learning objective.

Example:

```text
reg:squarederror
→ regression with squared error
```

# 15. XGBOOST CORE MATH

XGBoost uses:

```text
Gradient + Hessian
```

Gradient:

```text
G = sum(gradients)
```

Hessian:

```text
H = sum(hessians)
```

Simplified leaf weight:

```text
w = -G / (H + lambda)
```

This is one of the important differences  
from basic Gradient Boosting.

# 16. SPLIT GAIN

For a potential split:

```text
    Parent
   /      \
Left     Right
```

Gain:

1/2 \* [

```text
G_left^2 / (H_left + lambda)

+

G_right^2 / (H_right + lambda)

-

G_parent^2 / (H_parent + lambda)
```

]

- gamma

The split with the highest positive gain  
is preferred.

# 17. CORE WORKING

XGBoost:

```text
Current predictions
       ↓
Calculate gradients
       ↓
Calculate Hessians
       ↓
Try possible tree splits
       ↓
Calculate split gain
       ↓
Choose useful splits
       ↓
Calculate leaf weights
       ↓
Add tree using learning rate
       ↓
Repeat
```

# 18. GRADIENT BOOSTING VS XGBOOST

Gradient Boosting:

```text
Current prediction
      ↓
Calculate residual / gradient
      ↓
Train tree
      ↓
Add tree
```

XGBoost:

```text
Current prediction
      ↓
Gradient + Hessian
      ↓
Optimize tree splits
      ↓
Regularization
      ↓
Calculate leaf weights
      ↓
Add tree
```

XGBoost adds:

1. Second-order optimization
2. Regularization
3. Better split optimization
4. Row/column subsampling
5. Engineering optimizations

# 19. INTERVIEW SUMMARY

XGBoost is an optimized and regularized  
gradient boosting algorithm.

Main ideas:

Gradient  
Hessian  
Tree-based boosting  
Regularization  
Shrinkage  
Split gain

Important parameters:

n_estimators  
learning_rate  
max_depth  
min_child_weight  
subsample  
colsample_bytree  
gamma  
reg_alpha  
reg_lambda